In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
# Load the data
df=pd.read_csv("dataset/housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [6]:
# Creating new column is income_cat
df['income_cat']=pd.cut(df['median_income'],bins=[0,1.5,3.0,4.5,6.0,np.inf],labels=[1,2,3,4,5])
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_cat
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,5
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,5
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,5
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,4
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,3


In [7]:
#Stratified shuffle split in scikit-learn
# Scikit-learn provides a built-in way to perform stratified sampling using StratifiedShuffleSplit.
from sklearn.model_selection import StratifiedShuffleSplit
split=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=40)
for train_index, test_index in split.split(df,df['income_cat']):
    start_train_set=df.loc[train_index]
    start_test_set=df.loc[test_index]

# check the training_set
# start_train_set.head(10)
# check the testing_set data
start_train_set.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_cat
16415,-121.26,37.88,42.0,465.0,93.0,256.0,93.0,3.1719,158300.0,INLAND,3
1279,-121.64,37.85,22.0,1999.0,415.0,967.0,320.0,4.4583,253900.0,INLAND,3
7741,-118.15,33.95,35.0,2753.0,702.0,1592.0,614.0,2.7875,209000.0,<1H OCEAN,2
4425,-118.24,34.07,27.0,223.0,80.0,249.0,82.0,1.6136,137500.0,<1H OCEAN,2
1283,-121.82,38.02,46.0,176.0,43.0,101.0,40.0,2.2361,93800.0,INLAND,2


In [105]:
# lets remove the income_cat column
for st in (start_train_set ,start_test_set):
    st.drop("income_cat",axis=1,inplace=True)

KeyError: "['income_cat'] not found in axis"

In [106]:
df=start_train_set.copy()
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
16415,-121.26,37.88,42.0,465.0,93.0,256.0,93.0,3.1719,158300.0,INLAND
1279,-121.64,37.85,22.0,1999.0,415.0,967.0,320.0,4.4583,253900.0,INLAND
7741,-118.15,33.95,35.0,2753.0,702.0,1592.0,614.0,2.7875,209000.0,<1H OCEAN
4425,-118.24,34.07,27.0,223.0,80.0,249.0,82.0,1.6136,137500.0,<1H OCEAN
1283,-121.82,38.02,46.0,176.0,43.0,101.0,40.0,2.2361,93800.0,INLAND
...,...,...,...,...,...,...,...,...,...,...
1221,-120.65,38.28,21.0,3095.0,681.0,1341.0,546.0,2.1382,104000.0,INLAND
12640,-121.45,38.53,34.0,1893.0,415.0,884.0,395.0,2.1679,75400.0,INLAND
5747,-118.27,34.17,48.0,1560.0,280.0,825.0,269.0,5.5118,354700.0,<1H OCEAN
16224,-121.33,37.98,36.0,3113.0,576.0,1746.0,544.0,3.4625,84600.0,INLAND


In [14]:
# Further Preprocessing technique
housing=start_train_set.drop("median_house_value",axis=1)
housing_labels=start_train_set["median_house_value"].copy()

In [15]:
# This computes the median for each numerical column and stores it in imputer.statistics_:
from sklearn.impute import SimpleImputer 
imputer= SimpleImputer(strategy="median")
housing_num=housing.select_dtypes(include=[np.number])

In [16]:
imputer.fit(housing_num)

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [17]:
imputer.statistics_

array([-118.48  ,   34.25  ,   29.    , 2129.    ,  436.    , 1168.    ,
        411.    ,    3.5341])

In [18]:
x=imputer.transform(housing_num)
x

array([[-121.26  ,   37.88  ,   42.    , ...,  256.    ,   93.    ,
           3.1719],
       [-121.64  ,   37.85  ,   22.    , ...,  967.    ,  320.    ,
           4.4583],
       [-118.15  ,   33.95  ,   35.    , ..., 1592.    ,  614.    ,
           2.7875],
       ...,
       [-118.27  ,   34.17  ,   48.    , ...,  825.    ,  269.    ,
           5.5118],
       [-121.33  ,   37.98  ,   36.    , ..., 1746.    ,  544.    ,
           3.4625],
       [-118.4   ,   34.    ,   37.    , ...,  751.    ,  259.    ,
           5.444 ]], shape=(16512, 8))

In [ ]:
# Handling categorical values.

In [109]:
# 1. Categorical Attributes
# Text columns like "ocean_proximity" are not free-form text but limited to a fixed set of values
# (e.g., "NEAR BAY" and "INLAND"). These are known as categorical attributes.

housing=pd.DataFrame(x,columns=housing_num.columns,index=housing_num.index)
housing['ocean_proximity']=df['ocean_proximity']
housing=housing[['ocean_proximity']]

housing

,ocean_proximity
16415,INLAND
1279,INLAND
7741,<1H OCEAN
4425,<1H OCEAN
1283,INLAND
...,...
1221,INLAND
12640,INLAND
5747,<1H OCEAN
16224,INLAND


In [110]:
set(housing['ocean_proximity'])

{'<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'}

In [111]:
# 2. Ordinal Encoding :Scikit-Learn's OrdinalEncoder can convert categories to numbers:
# from sklearn.preprocessing import OrdinalEncoder
 
# ordinal_encoder = OrdinalEncoder()
# housing_cat = ordinal_encoder.fit_transform(housing)
# housing_cat=pd.DataFrame(housing_cat,columns=housing.columns,index=housing.index)
# housing_cat

In [112]:
# 3. One-Hot Encoding: for unordered categories, one-hot encoding is a better choice. 
# It creates one binary column per category.
from sklearn.preprocessing import OneHotEncoder

one_hot=OneHotEncoder()
housing_one=one_hot.fit_transform(housing)


In [113]:
one_hot.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [117]:
housing_one.toarray()

array([[0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       ...,
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.]], shape=(16512, 5))

In [119]:
housing_one=pd.DataFrame(housing_one.toarray(),columns=['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],index=housing.index)
housing_one

,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
16415,0.0,1.0,0.0,0.0,0.0
1279,0.0,1.0,0.0,0.0,0.0
7741,1.0,0.0,0.0,0.0,0.0
4425,1.0,0.0,0.0,0.0,0.0
1283,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...
1221,0.0,1.0,0.0,0.0,0.0
12640,0.0,1.0,0.0,0.0,0.0
5747,1.0,0.0,0.0,0.0,0.0
16224,0.0,1.0,0.0,0.0,0.0


# Method	Use When	Output Type
* OrdinalEncoder	Categories have an order	2D NumPy array
* OneHotEncoder:    Categories are unordered,   sparse, or dense